In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
import torch.nn.functional as F
import shap 
from functools import reduce
import random
from sklearn.model_selection import train_test_split

In [2]:
# Mapping for 20 standard amino acids
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

In [3]:
# Dataset class
class ProteinDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

        lengths = [len(seq) for seq in sequences]
        min_len = min(lengths)
        max_len = max(lengths)

        if max_len - min_len > 2:
            raise ValueError(f"Sequences vary too much in length! Found lengths: {set(lengths)}")

        self.seq_len = max_len  # allow minor difference (pad shorter sequences)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]

        onehot = np.zeros((20, self.seq_len), dtype=np.float32)
        for i, aa in enumerate(seq):
            if i >= self.seq_len:  # (safety) but should never happen
                break
            if aa in AA_TO_INDEX:
                onehot[AA_TO_INDEX[aa], i] = 1.0

        return torch.tensor(onehot), torch.tensor(label, dtype=torch.float32)



In [4]:
class ProteinTransformer(nn.Module):
    def __init__(self, input_dim=20, d_model=128, nhead=4, num_layers=2,
                 max_len=6000):                      # upper bound
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=256, dropout=0.1,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # pre-compute positional encoding once
        pe = self._make_pe(d_model, max_len)
        self.register_buffer("pos_enc", pe)          # NOT a parameter

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Dropout(0.5),                        # small reg
            nn.Linear(d_model, 1)                  # logits
        )

    @staticmethod
    def _make_pe(d_model, length, device="cpu"):
        pos = torch.arange(length).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(1e4)/d_model))
        pe  = torch.zeros(length, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe                                  # (L,d_model)

    def forward(self, x):                          # x (B,C,L)
        x = x.permute(0, 2, 1)                     # (B,L,C)
        x = self.embedding(x)                      # (B,L,d_model)
        L = x.size(1)
        x = x + self.pos_enc[:L, :].unsqueeze(0)   # add PE
        x = self.encoder(x)                        # (B,L,d_model)
        x = x.permute(0, 2, 1)                     # (B,d_model,L)
        return self.classifier(x).squeeze(-1)      # logits


In [5]:

class Wrapped(nn.Module):
    """
    Tiny adapter so SHAP sees output shape (B,1) instead of (B,)
    """
    def __init__(self, base): super().__init__(); self.base = base
    def forward(self, x):     return self.base(x).unsqueeze(1)

def shap_per_residue(model, train_ds, val_ds,
                     background_size=100, explain_samples=200,
                     per_gene_lengths=None, gene_names=None,
                     device="cuda"):
    """
    Returns a DataFrame with columns:
       sample_idx , label , importance_full , importance_<gene1> , …
    * importance_… columns hold np.ndarray vectors with per-residue |SHAP|.
    """
    model = model.to(device).eval()

    # ---- (1) pick SHAP background from *training* set ------------
    B = min(background_size, len(train_ds))
    bg_idx = random.sample(range(len(train_ds)), B)
    background = torch.stack([train_ds[i][0] for i in bg_idx]).to(device)

    explainer = shap.DeepExplainer(Wrapped(model), [background])

    # ---- (2) pick validation samples to explain ------------------
    E = min(explain_samples, len(val_ds))
    samp_idx = random.sample(range(len(val_ds)), E)
    xs = torch.stack([val_ds[i][0] for i in samp_idx]).to(device)
    ys = [val_ds[i][1] for i in samp_idx]

    # ---- (3) compute SHAP ---------------------------------------
    sv  = explainer.shap_values([xs], check_additivity=False)[0]   # (E,C,L)
    imp = np.abs(sv).sum(axis=1)                                   # (E,L)

    out = {
        "sample_idx"      : samp_idx,
        "label"           : [int(y) for y in ys],
        "importance_full" : list(imp),
    }

    # ---- (4) slice per-gene chunks if needed ---------------------
    if per_gene_lengths is not None:
        cuts = np.cumsum([0] + per_gene_lengths)                  # e.g. [0,2517,4545]
        for gi, g in enumerate(gene_names):
            out[f"importance_{g}"] = [imp[n, cuts[gi]:cuts[gi+1]] for n in range(E)]

    return pd.DataFrame(out)

In [6]:
def dedup_and_save_indices(ds, name, out_dir="data/latest/results/dedup", force=False):
    """
    Deduplicate a dataset by exact sequence+label.
    Saves indices to .npy and logs the reduction to a CSV file.

    If indices already exist, they are loaded instead (unless force=True).
    Log is overwrite-safe: updates existing row for the dataset.
    """
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"transformer_{name}_dedup_indices.npy"
    log_path = out_dir / "transformer_dedup_log.csv"

    # --- (1) use cache if available ---
    if out_path.exists() and not force:
        uniq_indices = np.load(out_path, allow_pickle=True).tolist()
        print(f"[{name}] using cached dedup indices ({len(uniq_indices)} sequences)")
        return uniq_indices

    # --- (2) compute fresh dedup ---
    uniq_indices, seen = [], set()
    for i in range(len(ds)):
        x, y = ds[i]
        key = (x.numpy().tobytes(), int(y))  # exact tensor bytes + label
        if key not in seen:
            seen.add(key)
            uniq_indices.append(i)

    np.save(out_path, uniq_indices)

    # --- (3) prepare log row ---
    new_row = {
        "Dataset": name,
        "Original_Size": len(ds),
        "Dedup_Size": len(uniq_indices),
        "Reduction": len(ds) - len(uniq_indices),
        "Fraction_Removed": round(1 - len(uniq_indices)/len(ds), 3)
    }

    # --- (4) write/overwrite log ---
    if log_path.exists():
        log_df = pd.read_csv(log_path)
        # remove existing row for this dataset if present
        log_df = log_df[log_df["Dataset"] != name]
        log_df = pd.concat([log_df, pd.DataFrame([new_row])], ignore_index=True)
        log_df.to_csv(log_path, index=False)
    else:
        pd.DataFrame([new_row]).to_csv(log_path, index=False)

    print(f"[{name}] deduplicated {len(ds)} → {len(uniq_indices)} "
          f"({len(ds)-len(uniq_indices)} removed, "
          f"{100*(1-len(uniq_indices)/len(ds)):.1f}% reduction)")

    return uniq_indices


In [7]:
def load_onehot_transformer(drug, run_dir="data/latest/results/prediction/transformer/transformer_models",
                            d_model=128, nhead=4, num_layers=2, max_len=6000):
    """
    Reload saved one-hot Transformer for a drug.
    Returns (model, seq_len).
    """
    # detect sequence length
    seqs = _build_drug_df(drug)["Protein_Sequence"].tolist()
    seq_len = max(len(s) for s in seqs)

    # always build with training-time max_len (6000)
    model = ProteinTransformer(
        input_dim=20, d_model=d_model, nhead=nhead,
        num_layers=num_layers, max_len=max_len
    ).to(device)

    # load weights, ignoring pos_enc mismatch
    model_path = Path(run_dir) / f"{drug}_transformer.pt"
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)   # ignore buffer mismatch
    model.eval()

    return model, seq_len



In [8]:
def compute_shap_transformer_onehot(drug, background_frac=1.0, explain_frac=1.0,
                            out_root="data/latest/results"):
    """
    Compute SHAP for saved one-hot transformer model, with cached dedup indices.
    Produces:
      - {drug}_transformer_shap_all.pkl   (exploratory: train+test)
      - {drug}_transformer_shap_test.pkl  (deployment: test-only)
    """
    # --- load trained model ---
    model, seq_len = load_onehot_transformer(drug)

    # --- rebuild train/test splits ---
    train_df, test_df = build_train_test_split(drug)

    train_ds = ProteinDataset(
        train_df["Protein_Sequence"].tolist(),
        (train_df["Phenotype"] == "R").astype(int).tolist()
    )
    test_ds = ProteinDataset(
        test_df["Protein_Sequence"].tolist(),
        (test_df["Phenotype"] == "R").astype(int).tolist()
    )
    full_ds = ProteinDataset(
        train_df["Protein_Sequence"].tolist() + test_df["Protein_Sequence"].tolist(),
        (train_df["Phenotype"] == "R").astype(int).tolist() + 
        (test_df["Phenotype"] == "R").astype(int).tolist()
    )

    out_path = Path(f"{out_root}/interpretability/transformer")
    out_path.mkdir(parents=True, exist_ok=True)

    # --- deduplicate (cached if exists) ---
    train_idx = dedup_and_save_indices(train_ds, f"{drug}_train")
    test_idx  = dedup_and_save_indices(test_ds,  f"{drug}_test")
    full_idx  = dedup_and_save_indices(full_ds,  f"{drug}_full")

    train_ds = torch.utils.data.Subset(train_ds, train_idx)
    test_ds  = torch.utils.data.Subset(test_ds,  test_idx)
    full_ds  = torch.utils.data.Subset(full_ds,  full_idx)

    # --- compute per-gene lengths & names ---
    gene_names = DRUG2GENES[drug]
    if len(gene_names) == 1:
        per_gene_lengths = [seq_len]
    else:
        # use _build_drug_df to recover per-gene sequence lengths
        df = _build_drug_df(drug)
        per_gene_lengths = [len(df[f"seq_{g}"].iloc[0]) for g in gene_names]

        # sanity check
        total_len = sum(per_gene_lengths)
        assert total_len == seq_len, \
            f"{drug}: expected {seq_len}, got {total_len}"

        # verbose logging
        lengths_str = ", ".join(f"{g}={L}" for g, L in zip(gene_names, per_gene_lengths))
        print(f"[per-gene lengths] {drug}: {lengths_str} (sum={total_len}, expected={seq_len})")

    # --- exploratory SHAP (train+test) ---
    bg_size = max(1, int(len(train_ds) * background_frac))
    ex_size = max(1, int(len(full_ds)  * explain_frac))
    shap_all = shap_per_residue(
        model=model, train_ds=train_ds, val_ds=full_ds,
        background_size=bg_size, explain_samples=ex_size,
        per_gene_lengths=per_gene_lengths if len(gene_names) > 1 else None,
        gene_names=gene_names if len(gene_names) > 1 else None,
        device=device
    )
    shap_all.to_pickle(out_path / f"{drug}_transformer_shap_all.pkl", protocol=4)

    # --- test-only SHAP ---
    ex_size = max(1, int(len(test_ds) * explain_frac))
    shap_test = shap_per_residue(
        model=model, train_ds=train_ds, val_ds=test_ds,
        background_size=bg_size, explain_samples=ex_size,
        per_gene_lengths=per_gene_lengths if len(gene_names) > 1 else None,
        gene_names=gene_names if len(gene_names) > 1 else None,
        device=device
    )
    shap_test.to_pickle(out_path / f"{drug}_transformer_shap_test.pkl", protocol=4)

    print(f"[done] {drug}: exploratory={len(full_ds)} | test={len(test_ds)} isolates")


In [13]:
# TODO = [ "ethambutol" # for transformer 100% is too much and exploding memory for most drugs. (rif)
# rif, eth, amc, cm,lev, mox, pnca, stm,inh,eth.  lets try 10%
TODO = ["ethambutol"]
device="cuda" if torch.cuda.is_available() else "cpu"
for drug in TODO:
    compute_shap_transformer_onehot(drug, background_frac=0.1, explain_frac=0.1)


[ethambutol_train] using cached dedup indices (1293 sequences)
[ethambutol_test] using cached dedup indices (499 sequences)
[ethambutol_full] using cached dedup indices (1532 sequences)
[per-gene lengths] ethambutol: embC=1094, embA=1094, embB=1098 (sum=3286, expected=3286)


/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/shap/explainers/_deep/deep_pytorch.py:243: UserWarning: unrecognized nn.Module: Flatten
  warnings.warn(f'unrecognized nn.Module: {module_type}')
/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/shap/explainers/_deep/deep_pytorch.py:243: UserWarning: unrecognized nn.Module: LayerNorm
  warnings.warn(f'unrecognized nn.Module: {module_type}')


[done] ethambutol: exploratory=1532 | test=499 isolates


In [9]:
# ------------------------------------------------------------------
#  Train, log curve *and* compute SHAP for ONE dataset
# ------------------------------------------------------------------
def train_eval_model(
        tag: str,                 # e.g. "moxifloxacin"
        df: pd.DataFrame,         # must have Protein_Sequence + Phenotype
        n_epochs:    int = 10,
        lr:          float = 1e-3,
        batch_size:  int = 32,
        device:      str  = "cuda" if torch.cuda.is_available() else "cpu",
        # ---- SHAP-specific opts -----------------------------------
        compute_shap:      bool   = True,
        background_size:   int    = 100,
        explain_samples:   int    = 200,
        per_gene_lengths:  list   = None,      # e.g. [2517,2028] or None
        gene_names:        list   = None,      # ["gyrA","gyrB"]  or None
):
    # 1) ── build ONE train/val split so we can reuse later ──────────────
    full_seqs   = df["Protein_Sequence"].tolist()
    full_labels = (df["Phenotype"] == "R").astype(int).tolist()          # ← ADD THIS
    
    full_ds = ProteinDataset(full_seqs, full_labels)
    
    idx = np.arange(len(full_ds))
    train_idx, val_idx = train_test_split(
        idx, test_size=0.2,
        stratify=full_labels,               # ← use full_labels here
        random_state=42
    )
    
    train_ds = torch.utils.data.Subset(full_ds, train_idx)
    val_ds   = torch.utils.data.Subset(full_ds, val_idx)


    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    # 2) ─── model / loss / optimiser --------------------------------
    model = ProteinTransformer().to(device)
    optimiser = optim.Adam(model.parameters(), lr=lr)
    

    # 2) ── class-imbalance weight --------------------------------------
    n_pos = sum(np.array(full_labels)[train_idx])
    n_neg = len(train_idx) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], device=device)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    curve_rows = []


    for ep in range(1, n_epochs + 1):
        # ----- training -------------------------------------------
        model.train(); running_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimiser.zero_grad()
            logits = model(X)
            loss   = criterion(logits, y)
            loss.backward(); optimiser.step()
            running_loss += loss.item() * X.size(0)
        epoch_loss = running_loss / len(train_ds)

        # ----- quick eval on VAL split ----------------------------
        model.eval(); preds, gold = [], []
        with torch.no_grad():
            for X, y in val_loader:
                X = X.to(device)
                logits = model(X)
                preds.extend(torch.sigmoid(logits).cpu().numpy())  # convert to prob
                gold.extend(y.numpy())
        epoch_auc = roc_auc_score(gold, preds)

        curve_rows.append({"Epoch": ep, "Loss": epoch_loss, "AUC": epoch_auc})
        print(f"{tag} | epoch {ep:02d} | loss {epoch_loss:.4f} | AUC {epoch_auc:.3f}")

    # 3) ─── save training artefacts ---------------------------------
    out_dir = Path("data/latest/results/prediction/transformer"); out_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(curve_rows).to_csv(out_dir / f"{tag}_training_curve.csv", index=False)
    torch.save(model.state_dict(), out_dir / f"transformer_models/{tag}_transformer.pt")

    # 4) ─── SHAP after training (optional) --------------------------  ### NEW
    if compute_shap:
        shap_df = shap_per_residue(
            model              = model,
            train_ds           = train_ds,   # same split used for training
            val_ds             = val_ds,     # held-out data
            background_size    = background_size,
            explain_samples    = explain_samples,
            per_gene_lengths   = per_gene_lengths,
            gene_names         = gene_names,
            device             = device,
        )
        out_path = Path("data/latest/results/interpretability/transformer"); out_path.mkdir(exist_ok=True)
        shap_df.to_pickle(out_path / f"{tag}_shap_per_residue.pkl", protocol=4)
        print("SHAP saved:", out_path / f"{tag}_shap_per_residue.pkl")

    # 5) ─── return 1-row summary for aggregation --------------------
    return pd.DataFrame({
        "Tag"        : [tag],
        "Final_AUC"  : [curve_rows[-1]["AUC"]],
        "Final_Loss" : [curve_rows[-1]["Loss"]],
        "Epochs"     : [n_epochs],
    }), model


In [10]:
# ─── CONFIG & DRUG→GENE MAP ─────────────────────────────────────────────
DATA_DIR = Path("./data/latest/sequence_data_csv")
AUC_DIR  = Path("data/latest/results/prediction/transformer");  AUC_DIR.mkdir(parents=True, exist_ok=True)

DRUG2GENES = {
    "rifampicin"  : ["rpoB"],
    "pyrazinamide": ["pncA"],
    "capreomycin" : ["tlyA"],
    "amikacin"    : ["eis"],

    "moxifloxacin": ["gyrB","gyrA"],
    "levofloxacin": ["gyrB","gyrA"],
    "isoniazid"   : ["katG","inhA"],
    "streptomycin": ["rpsL","gid"],
    "ethambutol"  : ["embC","embA","embB"],
    "ethionamide" : ["ethA","ethR","inhA"],
}


# global knobs (edit once)
N_EPOCHS        = 20
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
BG_SIZE         = 150      # SHAP background samples
EXPL_SAMPLES    = 250      # SHAP explain samples
BATCH_SIZE      = 32
LR              = 5e-4


In [ ]:

from sklearn.model_selection import train_test_split

def build_train_test_split(drug:str, test_frac=0.2, seed=42):
    df = _build_drug_df(drug) 
    train_df, test_df = train_test_split(
        df, test_size=test_frac, random_state=seed,
        stratify=df["Phenotype"]
    )
    return train_df, test_df


# ─── 1.  FIND THE RIGHT CSV FOR (gene, drug) ───────────────────────────
def _csv_path(gene:str, drug:str)->Path:
    special = DATA_DIR / f"{gene}_{drug.upper()}_combined_sequence_data.csv"
    generic = DATA_DIR / f"{gene}_combined_sequence_data.csv"
    if special.exists(): return special
    if generic.exists(): return generic
    raise FileNotFoundError(f"Missing CSV for {gene} ({drug})")


# ─── 2.  BUILD ONE DATAFRAME PER DRUG (concatenated sequences) ─────────
def _build_drug_df(drug:str)->pd.DataFrame:
    gene_dfs = []
    for g in DRUG2GENES[drug]:
        df = pd.read_csv(_csv_path(g, drug))
        df = df[(df["Frameshift_Mutation"]==0) &
                (df["Phenotype"].isin(["R","S"]))].copy()
        df = df[["Filename","Protein_Sequence","Phenotype"]]
        df = df.rename(columns={"Protein_Sequence": f"seq_{g}"})
        gene_dfs.append(df)

    # inner-join on Filename & Phenotype so we keep only isolates
    # present in *all* genes
    def _merge(a,b):
        return pd.merge(a,b,on=["Filename","Phenotype"], how="inner")
    merged = reduce(_merge, gene_dfs)

    # sanity: no conflicting phenotypes
    assert merged["Phenotype"].nunique() <= 2

    # concatenate sequences in gene order
    merged["Protein_Sequence"] = merged[[f"seq_{g}" for g in DRUG2GENES[drug]]].agg("".join, axis=1)
    # merged = merged[["Filename","Protein_Sequence","Phenotype"]]
    return merged

In [12]:
from sklearn.metrics import roc_auc_score

def evaluate_auc(model, df, device="cpu"):
    ds = ProteinDataset(
        df["Protein_Sequence"].tolist(),
        (df["Phenotype"]=="R").astype(int).tolist()
    )
    loader = DataLoader(ds, batch_size=32, shuffle=False)
    preds, gold = [], []
    with torch.no_grad():
        for X,y in loader:
            X = X.to(device)
            logits = model(X)
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            gold.extend(y.numpy())
    return roc_auc_score(gold, preds)


In [ ]:
# ─── 3.  WRAPPER THAT CALLS EXISTING train_eval_model ─────────────
def run_transformer_for_drug(drug:str, n_epochs:int=10):
    print(f"\n=== {drug.upper()} ({', '.join(DRUG2GENES[drug])}) ===")
    train_df, test_df = build_train_test_split(drug)
    df = _build_drug_df(drug)
    max_len = df["Protein_Sequence"].str.len().max()
    # per-gene residue lengths for SHAP slicing
    per_gene_lengths = [len(df[f"seq_{g}"].iloc[0]) for g in DRUG2GENES[drug]]
    gene_names       = DRUG2GENES[drug]

    print(f"\n=== {drug.upper()} ({', '.join(gene_names)}) | "
          f"{len(df)} isolates | lengths {per_gene_lengths}")

    # drop the helper columns now—they’re no longer needed
    df = df[["Filename", "Protein_Sequence", "Phenotype"]]

    print(f"{drug.upper()} | isolates={len(df)} | lengths={per_gene_lengths}")
    
    summary_df,model = train_eval_model(
        tag               = drug,
        df                = train_df,
        n_epochs          = N_EPOCHS,
        lr                = LR,
        batch_size        = BATCH_SIZE,
        device            = DEVICE,
        # SHAP parameters
        compute_shap      = True,
        background_size   = BG_SIZE,
        explain_samples   = EXPL_SAMPLES,
        per_gene_lengths  = per_gene_lengths if len(gene_names) > 1 else None,
        gene_names        = gene_names      if len(gene_names) > 1 else None,
    )
    train_auc = evaluate_auc(model, train_df, device=DEVICE)
    test_auc  = evaluate_auc(model, test_df,  device=DEVICE)
    print(f"{drug}: Train AUC={train_auc:.3f} | Test AUC={test_auc:.3f}")

    results = pd.DataFrame({
        "Drug": [drug],
        "Train_AUC": [train_auc],
        "Test_AUC": [test_auc],
    })
    results.to_csv(
        AUC_DIR / f"{drug}_train_test_auc.csv",
        index=False
    )
    return summary_df



# ─── run the full panel of drugs ────────────────────────────

drug_list = ['rifampicin', 'pyrazinamide','capreomycin', 'amikacin','isoniazid','ethionamide','streptomycin','ethambutol','moxifloxacin','levofloxacin']

all_dfs   = []
for d in tqdm(drug_list):
    if d not in DRUG2GENES:
        print(f"[skip] {d}"); continue
    try:
        all_dfs.append(run_transformer_for_drug(d))
    except FileNotFoundError as e:
        print("‣", e)

if all_dfs:
    pd.concat(all_dfs, ignore_index=True)\
      .to_csv("data/latest/results/prediction/transformer_summary_auc_scores.csv", index=False)
    print("saved transformer_summary_auc_scores.csv")

## step 7: precision and recall

In [ ]:
# pr_from_shap.py
# ------------------------------------------------------------
import pandas as pd, numpy as np
from pathlib import Path
from functools import reduce

# ─── 0. Global knobs ---------------------------------------------------
K_VALUES      = [1, 5, 10]
ALLOWED_CONF  = ['1) Assoc w R', '2) Assoc w R - Interim']

DATA_DIR      = Path("data")                                # root
SHAP_DIR      = DATA_DIR / "latest/results/interpretability/transformer"                     # *.pkl here
CATALOG_CSV   = DATA_DIR / "filtered_variants_output.csv"
OUT_DIR       = SHAP_DIR/ "pr_tables"; OUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── 0-b.  Drug → genes map 
DRUG2GENES = {
    "rifampicin"  : ["rpoB"],
    "pyrazinamide": ["pncA"],
    "capreomycin" : ["tlyA"],
    "amikacin"    : ["eis"],
    "moxifloxacin": ["gyrA","gyrB"],
    "levofloxacin": ["gyrA","gyrB"],
    "isoniazid"   : ["katG","inhA"],
    "streptomycin": ["rpsL","gid"],
    "ethambutol"  : ["embC","embA","embB"],   # corrected order
    "ethionamide" : ["ethA","ethR","inhA"],
}

# ─── 1. Load WHO catalogue once ---------------------------------------
catalog = pd.read_csv(CATALOG_CSV)
catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) - 1  # zero-based

def bona_fide(cat_rows):
    m = cat_rows["confidence"].isin(ALLOWED_CONF) & (cat_rows["intersectional"] == True)
    return set(cat_rows.loc[m, "aa_pos_0idx"])

# def exclusion(cat_rows):
#     m = (cat_rows["confidence"] == "3) Uncertain significance") | (cat_rows["intersectional"] != True)
#     return set(cat_rows.loc[m, "aa_pos_0idx"])
def exclusion(cat_rows):
    mask_unc   = cat_rows["confidence"] == "3) Uncertain significance"
    mask_not_i = cat_rows["intersectional"] != True
    excl = set(cat_rows.loc[mask_unc | mask_not_i, "aa_pos_0idx"])
    return excl - bona_fide(cat_rows)          # ← keep bona-fide sites

def rows_for_gene(g):   # convenience
    return catalog[catalog["gene"].str.lower() == g.lower()]

# ─── 2. helper: rank residues by max|SHAP| across isolates -------------
def rank_by_max_abs(shap_df, col):
    stacks = np.stack([np.asarray(v).squeeze() for v in shap_df[col]], axis=0)  # (N,L)
    maximp = np.abs(stacks).max(axis=0)
    return (pd.DataFrame({"Residue_Position": np.arange(len(maximp)),
                          "MaxAbsSHAP": maximp})
            .sort_values("MaxAbsSHAP", ascending=False)
            .reset_index(drop=True))

def greedy_topk(rank_df, k, exclude):
    chosen=[]
    for pos in rank_df["Residue_Position"]:
        if pos in exclude:   # skip excluded sites
            continue
        chosen.append(pos)
        if len(chosen)==k:
            break
    return chosen

# ─── 3. driver per drug -----------------------------------------------
all_rows = []
for drug, genes in DRUG2GENES.items():
    shap_pkl = SHAP_DIR / f"{drug}_transformer_shap_all.pkl"
    if not shap_pkl.exists():
        print(f"[skip] {drug}: SHAP file missing."); continue

    shap_df = pd.read_pickle(shap_pkl)
        
    print(f"\n{drug.upper()}  (genes: {', '.join(genes)})")

    for g in genes:
        col_name = f"importance_{g}" if len(genes)>1 else "importance_full"
        if col_name not in shap_df.columns:
            print(f"  [warn] {col_name} col not found → skip"); continue

        rank_df    = rank_by_max_abs(shap_df, col_name)
        cat_rows   = rows_for_gene(g)
        gold_sites = bona_fide(cat_rows)
        excl_sites = exclusion(cat_rows)
        n_true     = len(gold_sites)

        for k in K_VALUES:
            topk   = greedy_topk(rank_df, k, exclude=excl_sites)
            k_eff  = len(topk)
            tp     = len(gold_sites & set(topk))
            prec   = tp / k_eff if k_eff else 0.0
            rec    = tp / n_true if n_true else 0.0
            f1     = 2*prec*rec/(prec+rec+1e-8) if (prec+rec) else 0.0

            hits = (cat_rows[cat_rows["aa_pos_0idx"].isin(topk) &
                             cat_rows["confidence"].isin(ALLOWED_CONF) &
                             (cat_rows["intersectional"] == True)]
                    .drop_duplicates("aa_pos_0idx")["variant"].tolist() or ["None"])

            all_rows.append({
                "drug":drug, "gene":g, "k_req":k, "k_eff":k_eff,
                "total_res_pos": n_true, "TP":tp,
                "precision":prec, "recall":rec, "F1":f1,
                "hit_variants": ", ".join(hits)
            })
        # optional: save per-gene ranking for further plots
        rank_df.to_csv(OUT_DIR / f"{drug}_{g}_ranked_SHAP.csv", index=False)

# ─── 4. aggregate & write CSV -----------------------------------------
summary = pd.DataFrame(all_rows)
summary.to_csv(OUT_DIR / "precision_recall_all_drugs_shapall.csv", index=False)
print("\ntable saved →", OUT_DIR/"precision_recall_all_drugs.csv")



RIFAMPICIN  (genes: rpoB)

PYRAZINAMIDE  (genes: pncA)

CAPREOMYCIN  (genes: tlyA)

AMIKACIN  (genes: eis)

MOXIFLOXACIN  (genes: gyrA, gyrB)

LEVOFLOXACIN  (genes: gyrA, gyrB)

ISONIAZID  (genes: katG, inhA)

STREPTOMYCIN  (genes: rpsL, gid)

ETHAMBUTOL  (genes: embC, embA, embB)

ETHIONAMIDE  (genes: ethA, ethR, inhA)

table saved → data/latest/results/interpretability/transformer/pr_tables/precision_recall_all_drugs.csv


In [ ]:
import pandas as pd
from pathlib import Path
import glob
import re          # ↳ to parse folder names like "katG_dim320"


def gather_run_metrics(runs_root="data/latest/results/prediction/transformer",
                       history_glob="*_training_curve.csv",
                       save_as="transformer_summary_auc_scores.csv",
                       keep_last=True):
    """
    Walk through <runs_root>/**/<history_glob>, pull the best (and last) AUC
    from each history file, return + save a combined DataFrame.

    Parameters
    ----------
    runs_root     : str | Path   – parent directory that holds the run-folders
    history_glob  : str          – pattern that matches the history csv files
    save_as       : str          – filename for the combined table
    keep_last     : bool         – also store the *last* epoch's AUC

    Returns
    -------
    pandas.DataFrame
       columns:  run_dir, gene, dim, best_auc, epoch_at_best
                 [last_auc, last_epoch]  (if keep_last=True)
    """
    runs_root = Path(runs_root)
    history_files = glob.glob(str(runs_root / "**" / history_glob),
                              recursive=True)

    if not history_files:
        raise FileNotFoundError(f"No history CSVs found under {runs_root}")

    rows = []
    # rx_folder = re.compile(r"(?P<drug>[A-Za-z0-9_]"))

    for csv_path in history_files:
        csv_path = Path(csv_path)
        drug_name = (Path(csv_path).stem).split('_')[0]
        print(drug_name)
        run_dir  = csv_path.parent

        # --- try to parse "gene_dimXYZ" from the run folder name -------------
        # m = rx_folder.search(run_dir.name)
        # gene = m.group("gene") if m else "unknown"
        # dim  = int(m.group("dim")) if m else -1

        df = pd.read_csv(csv_path)
        best_row  = df.loc[df["AUC"].idxmax()]
        best_auc  = best_row["AUC"]
        best_ep   = int(best_row["Epoch"])

        entry = {
            "drug"         : drug_name,
            "best_auc"     : best_auc,
            "epoch_at_best": best_ep,
        }

        # optionally keep the very last epoch
        if keep_last:
            entry["last_auc"]   = df["AUC"].iloc[-1]
            entry["last_epoch"] = int(df["Epoch"].iloc[-1])

        rows.append(entry)

    # ----------------------------------------------------
    master = pd.DataFrame(rows).sort_values(["drug"])
    out_file = runs_root / save_as
    master.to_csv(out_file, index=False)
    print(f"combined {len(rows)} runs  →  {out_file}")

    return master


summary = gather_run_metrics(runs_root="data/latest/results/prediction/transformer")   # defaults are fine
print(summary.head())

        

In [1]:
import pandas as pd
from pathlib import Path
import glob
import re


def gather_run_metrics(runs_root="data/latest/results/prediction/transformer",
                       history_glob="*_train_test_auc.csv",
                       save_as="transformer_train_test_auc.csv",
                       keep_last=True):
    """
    Walk through <runs_root>/**/<history_glob>, extract train + test aucs



    Returns
    -------
    pandas.DataFrame with columns:
        run_dir, gene, dim, dim_type, best_auc, epoch_at_best,
        [last_auc, last_epoch]
    """
    runs_root = Path(runs_root)
    history_files = glob.glob(str(runs_root / history_glob), recursive=True)

    if not history_files:
        raise FileNotFoundError(f"No history CSVs found under {runs_root}")

    rows = []

    # Pattern: captures 'gene' and the (type+dim) suffix
    rx_folder = re.compile(r"(?P<drug>[A-Za-z0-9_])")

    for csv_path in history_files:
        csv_path = Path(csv_path)
        run_dir = csv_path.parent

        m = rx_folder.match(run_dir.name)
        if not m:
            print(f"[!] Skipping unrecognized folder: {run_dir.name}")
            continue

        # drug = m.group("drug")

        df = pd.read_csv(csv_path)
        drug = df['Drug'][0]
        if df.empty or "Test_AUC" not in df.columns:
            print(f"[!] Skipping malformed CSV: {csv_path}")
            continue

        # best_row = df.loc[df["val_auc"].idxmax()]
        entry = {
            "run_dir": run_dir.name,
            "drug": drug,
            "train_auc": df['Train_AUC'][0],
            "test_auc": df['Test_AUC'][0]
            # "best_auc": best_row["val_auc"],
            # "epoch_at_best": int(best_row["epoch"]),
        }

        # if keep_last:
        #     entry["last_auc"] = df["val_auc"].iloc[-1]
        #     entry["last_epoch"] = int(df["epoch"].iloc[-1])

        rows.append(entry)

    master = pd.DataFrame(rows).sort_values(["drug"])
    out_file = runs_root / save_as
    master.to_csv(out_file, index=False)
    print(f"Combined {len(rows)} runs → {out_file}")

    return master


# Example call
summary = gather_run_metrics(runs_root="data/latest/results/prediction/transformer",
                             save_as="transformer_train_test_auc.csv")
print(summary.head())



Combined 10 runs → data/latest/results/prediction/transformer/transformer_train_test_auc.csv
       run_dir         drug  train_auc  test_auc
0  transformer     amikacin   0.498467  0.500378
1  transformer  capreomycin   0.501571  0.488947
2  transformer   ethambutol   0.651325  0.639560
3  transformer  ethionamide   0.402457  0.405253
4  transformer    isoniazid   0.899706  0.886879
